# Custom Voice Model with Tacotron 2 and WaveGlow

by Tobias Erbacher

To generate the Mel-spectrograms that Tacotron 2 uses as input, we first need to load the librosa package and import various other libraries. The audio files are uploaded for training purposes to my personal Google Drive.

In [1]:
from google.colab import drive
drive.mount("/content/drive")

import torch
from IPython.display import Audio
import tensorflow as tf
GPU_AVAILABLE = torch.cuda.is_available()

!pip install unidecode

if GPU_AVAILABLE:
  try:
    import librosa
  except:
    !pip install librosa
    import librosa
  import nltk
  nltk.download("punkt_tab")
  from nltk.tokenize import word_tokenize
  import re
  from torch.utils.data import Dataset
  from torch.utils.data import DataLoader

  !git clone https://github.com/NVIDIA/tacotron2.git
  !wget https://pytorch.org/assets/deep-learning-examples/tacotron2_statedict.pt
  !git clone https://github.com/NVIDIA/waveglow.git

  import sys
  sys.path.append("/content/drive/MyDrive/ATML_NLP_Assignment/")
  #sys.path.append("/content/tacotron2")
  #sys.path.append("/content/waveglow")
  #from tacotron2.model import Tacotron2
  #from tacotron2.loss_function import Tacotron2Loss
  #from waveglow.glow import WaveGlow

  # The next two files are taken from NVidia's github and need to be placed in the Colab directory. They are immported via the Google Drive.
  import Model
  import Loss_Function
else:
  print("Cannot train this model without a GPU.")

import os
import math
import pickle

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.5/235.5 kB 7.8 MB/s eta 0:00:00


[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


Cloning into 'tacotron2'...
remote: Enumerating objects: 412, done.
remote: Total 412 (delta 0), reused 0 (delta 0), pack-reused 412 (from 1)
Receiving objects: 100% (412/412), 2.70 MiB | 7.04 MiB/s, done.
Resolving deltas: 100% (203/203), done.
--2025-01-05 14:57:04--  https://pytorch.org/assets/deep-learning-examples/tacotron2_statedict.pt
Resolving pytorch.org (pytorch.org)... 185.199.108.153, 185.199.109.153, 185.199.110.153, ...
Connecting to pytorch.org (pytorch.org)|185.199.108.153|:443... connected.
HTTP request sent, awaiting response... 404 Not Found
2025-01-05 14:57:04 ERROR 404: Not Found.

Cloning into 'waveglow'...
remote: Enumerating objects: 196, done.
remote: Counting objects: 100% (6/6), done.
remote: Compressing objects: 100% (6/6), done.
remote: Total 196 (delta 2), reused 2 (delta 0), pack-reused 190 (from 1)
Receiving objects: 100% (196/196), 437.57 KiB | 1.38 MiB/s, done.
Resolving deltas: 100% (108/108), done.


In [2]:
# This is taken from
# https://github.com/NVIDIA/DeepLearningExamples/tree/master/PyTorch/SpeechSynthesis/Tacotron2/tacotron2/text
valid_symbols = [
  'AA', 'AA0', 'AA1', 'AA2', 'AE', 'AE0', 'AE1', 'AE2', 'AH', 'AH0', 'AH1', 'AH2',
  'AO', 'AO0', 'AO1', 'AO2', 'AW', 'AW0', 'AW1', 'AW2', 'AY', 'AY0', 'AY1', 'AY2',
  'B', 'CH', 'D', 'DH', 'EH', 'EH0', 'EH1', 'EH2', 'ER', 'ER0', 'ER1', 'ER2', 'EY',
  'EY0', 'EY1', 'EY2', 'F', 'G', 'HH', 'IH', 'IH0', 'IH1', 'IH2', 'IY', 'IY0', 'IY1',
  'IY2', 'JH', 'K', 'L', 'M', 'N', 'NG', 'OW', 'OW0', 'OW1', 'OW2', 'OY', 'OY0',
  'OY1', 'OY2', 'P', 'R', 'S', 'SH', 'T', 'TH', 'UH', 'UH0', 'UH1', 'UH2', 'UW',
  'UW0', 'UW1', 'UW2', 'V', 'W', 'Y', 'Z', 'ZH'
]
_pad        = '_'
_punctuation = '!\'(),.:;? '
_special = '-'
_letters = 'ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz'
_arpabet = ['@' + s for s in valid_symbols]
symbols = [_pad] + list(_special) + list(_punctuation) + list(_letters) + _arpabet

Now, we can load the voice samples to generate all the Mel-spectrograms.

In [3]:
data_path = "/content/drive/MyDrive/ATML_NLP_Assignment/1min_samples/data_samples_characterTokens.pkl"
vocab_path = "/content/drive/MyDrive/ATML_NLP_Assignment/1min_samples/vocab_characters.pkl"

if os.path.exists(data_path) and os.path.exists(vocab_path):
  with open(data_path, "rb") as f:
    data_samples = pickle.load(f)
  with open(vocab_path, "rb") as f:
    vocab = pickle.load(f)
else:
  SAMPLES_FOLDER_PATH = "/content/drive/MyDrive/ATML_NLP_Assignment/1min_samples/"
  SAMPLING_RATE = 22050
  N_FFT = 1024
  N_MEL = 80
  HOP_LENGTH = 256
  MEL_SPECTROGRAM_WINDOW_SIZE = 0.025
  MEL_SPECTROGRAM_HOP_SIZE = 0.01
  MEL_SPECTROGRAM_START_TIME = 0

  W = int(math.ceil(MEL_SPECTROGRAM_WINDOW_SIZE * SAMPLING_RATE))
  D = int(math.ceil(MEL_SPECTROGRAM_HOP_SIZE * SAMPLING_RATE))
  T = int(math.ceil(MEL_SPECTROGRAM_START_TIME * SAMPLING_RATE))

  data_samples = []
  vocab = {element : idx for idx, element in enumerate(symbols)}
  if GPU_AVAILABLE:
    for file in os.listdir(SAMPLES_FOLDER_PATH):
      if file.endswith(".wav"):
        y, sr = librosa.load(SAMPLES_FOLDER_PATH + file, sr=44100)
        y = librosa.to_mono(y)
        y = librosa.resample(y, orig_sr=sr, target_sr=22050)
        sr = 22050
        mel_spectrogram = librosa.feature.melspectrogram(y=y, sr=sr, n_fft=N_FFT, win_length=W, hop_length=D, n_mels=N_MEL)
        with open(SAMPLES_FOLDER_PATH + file[:-4]+".txt", "r") as file:
          text = file.read().lower()
          text = re.sub(r'[^a-zA-Z\s]', '', text)
          text = re.sub(r'\s+', ' ', text).strip()
          tokenized_text = word_tokenize(text)
          with open(SAMPLES_FOLDER_PATH + file[:-4]+".txt", "r") as file:
            text = list(file.read())
            tokenized_indices = [vocab.get(character, vocab[" "]) for character in text]

        data = (tokenized_indices, mel_spectrogram)
        data_samples.append(data)
    with open(data_path, "wb") as f:
      pickle.dump(data_samples, f)
    with open(vocab_path, "wb") as f:
      pickle.dump(vocab, f)
  else:
    print("Without a GPU this will take a while to train and is therefore aborted.")

Next, we can create the actual dataset:

In [4]:
class Tacotron2Dataset(Dataset):
  def __init__(self, data): # data: List of tuples (filename, tokenized_transcript, mel_spectrogram).
    self.data = data

  def __len__(self):
    return len(self.data)

  def __getitem__(self, idx):
    tokenized_transcript, mel_spectrogram = self.data[idx]

    mel_spectrogram = torch.tensor(mel_spectrogram, dtype=torch.float32)
    tokenized_transcript = torch.tensor(tokenized_transcript, dtype=torch.long)

    return {
      "tokenized_transcript": tokenized_transcript,
      "mel_spectrogram": mel_spectrogram,
    }

dataset = Tacotron2Dataset(data_samples)

Let us set some hyperparameters, the values are taken to be common values:

In [5]:
batch_size = 16
shuffle_dataset = True
learning_rate = 1e-4
step_size = 500
num_epochs = 500
gamma = 0.5

mask_padding = True
n_mel_channels = 80
n_symbols = len(symbols)
symbols_embedding_dim = 512
encoder_kernel_size = 5
encoder_n_convolutions = 3
encoder_embedding_dim = 512
attention_rnn_dim = 1024
attention_dim = 128
attention_location_n_filters = 32
attention_location_kernel_size = 31
n_frames_per_step = 1
decoder_rnn_dim = 1024
prenet_dim = 256
max_decoder_steps = 1000
gate_threshold = 0.5
p_attention_dropout = 0.1
p_decoder_dropout = 0.1
postnet_embedding_dim = 512
postnet_kernel_size = 5
postnet_n_convolutions = 5
decoder_no_early_stopping = True

In every batch, each element has to have the same length of spectrograms and tokenized transcripts. To this end, we use a collate function.

In [6]:
def pad_tacotron2(batch):
  spectrograms = [element["mel_spectrogram"] for element in batch]
  transcripts = [element["tokenized_transcript"] for element in batch]

  spectrogram_lengths = [element.shape[1] for element in spectrograms]
  max_spectrogram_length = max(spectrogram_lengths)
  padded_spectrograms = torch.zeros((len(spectrograms), spectrograms[0].shape[0], max_spectrogram_length))
  for idx, spectrogram in enumerate(spectrograms):
    padded_spectrograms[idx, :, :spectrogram.shape[1]] = spectrogram

  transcript_lengths = [len(element) for element in transcripts]
  max_transcript_length = max(transcript_lengths)
  padded_transcripts = torch.zeros((len(transcripts), max_transcript_length), dtype=torch.long)
  for idx, transcript in enumerate(transcripts):
    padded_transcripts[idx, :len(transcript)] = transcript

  return {
    "mel_spectrogram": padded_spectrograms,
    "tokenized_transcript": padded_transcripts,
    "mel_lengths": torch.tensor(spectrogram_lengths),
    "token_lengths": torch.tensor(transcript_lengths),
    "max_mel_length" : max_spectrogram_length
  }

Next, we initiate the dataloader and the pre-trained model.

In [7]:
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=shuffle_dataset, collate_fn=pad_tacotron2)

del data_samples
torch.cuda.empty_cache()

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

tacotron2_params = (
  mask_padding,
  n_mel_channels,
  n_symbols,
  symbols_embedding_dim,
  encoder_kernel_size,
  encoder_n_convolutions,
  encoder_embedding_dim,
  attention_rnn_dim,
  attention_dim,
  attention_location_n_filters,
  attention_location_kernel_size,
  n_frames_per_step,
  decoder_rnn_dim,
  prenet_dim,
  max_decoder_steps,
  gate_threshold,
  p_attention_dropout,
  p_decoder_dropout,
  postnet_embedding_dim,
  postnet_kernel_size,
  postnet_n_convolutions,
  decoder_no_early_stopping
)

tacotron2 = Model.Tacotron2(*tacotron2_params).to(DEVICE)
tacotron2.train()

# Freeze parameters to avoid overfitting (dataset is small).
for param in tacotron2.encoder.parameters():
  param.requires_grad = False

loss_function = Loss_Function.Tacotron2Loss()
optimizer = torch.optim.Adam(tacotron2.parameters(), lr=learning_rate)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=step_size, gamma=gamma)

Now, we train the model.

In [8]:
for epoch in range(num_epochs):
  tacotron2.train()
  epoch_loss = 0

  for batch in dataloader:
    optimizer.zero_grad()
    inputs, targets = batch["tokenized_transcript"].to(device=DEVICE, dtype=torch.long), batch["mel_spectrogram"].to(device=DEVICE, dtype=torch.float32)

    input_lengths, target_lengths, max_length = batch["token_lengths"].to(DEVICE), batch["mel_lengths"].to(DEVICE), batch["max_mel_length"]

    input_lengths, sorted_indices = input_lengths.sort(0, descending=True)
    inputs = inputs[sorted_indices]

    target_lengths, sorted_target_indices = target_lengths.sort(0, descending=True)
    targets = targets[sorted_target_indices]

    model_input = (inputs, input_lengths, targets, max_length, target_lengths)

    outputs = tacotron2.forward(model_input)
    loss = loss_function(outputs, targets)

    loss.backward()
    optimizer.step()

    epoch_loss += loss.item()

    del batch
    torch.cuda.empty_cache()

  scheduler.step()
  print(f"Epoch {epoch + 1}/{num_epochs}, Loss: {epoch_loss / len(dataloader)}")

torch.save(tacotron2.state_dict(), "/content/drive/MyDrive/ATML_NLP_Assignment/model/fine_tuned_tacotron2_charTokens.pt")

OutOfMemoryError: CUDA out of memory. Tried to allocate 20.00 MiB. GPU 0 has a total capacity of 14.75 GiB of which 3.06 MiB is free. Process 14767 has 14.74 GiB memory in use. Of the allocated memory 14.60 GiB is allocated by PyTorch, and 19.63 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

Unfortunately, when trying to train the model, we are running into computational issues because the GPU RAM seems to be filling up and interrupts the training.

---

Next, we could let the model say something via the WaveGlow mel spectrogram to audio functionality.

In [ ]:
INPUT = "Hello, this is an example sentence."
TOKENIZED = [vocab.get(token, vocab["<UNK>"]) for token in word_tokenize(INPUT.lower())]
INPUT_TENSOR = torch.tensor(TOKENIZED, dtype=torch.long).unsqueeze(0).to(DEVICE)
tacotron2.eval()
with torch.no_grad():
  MEL_OUTPUT, MEL_LENGTH, MEL_ALIGNMENTS = tacotron2(INPUT_TENSOR)

MEL_SPECTROGRAM = MEL_OUTPUT.squeeze(0).to(DEVICE)

In [ ]:
WAVEGLOW = torch.hub.load('NVIDIA/DeepLearningExamples:torchhub', 'nvidia_waveglow', model_math='fp32').to(DEVICE)
WAVEGLOW.eval()
with torch.no_grad():
    AUDIO = WAVEGLOW.infer(MEL_SPECTROGRAM)

AUDIO_NUMPY = AUDIO.cpu().numpy()

In [ ]:
Audio(AUDIO_NUMPY, rate=22050)